# Directory Contents Explorer

이 노트북은 현재 작업 폴더에 어떤 파일과 폴더가 들어있는지 한 번에 확인하기 위한 도구입니다.

- 별도 경로 입력 없이 `TARGET_DIR = Path.cwd()` 기준으로 스캔합니다.
- 노트북을 `/Users/lee/Github Clone/testcode`에서 열면 해당 폴더가 자동 대상이 됩니다.
- 숨김 파일, 재귀 탐색 깊이, 미리보기 개수는 설정 셀에서 조정할 수 있습니다.

## 1. 설정

아래 셀을 먼저 실행하세요. 대부분은 기본값 그대로 쓰면 됩니다.

In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import datetime
from pathlib import Path
import mimetypes
import os

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# 기본값: Jupyter가 실행 중인 현재 폴더를 스캔합니다.
TARGET_DIR = Path.cwd().resolve()

# 옵션
INCLUDE_HIDDEN = True        # .git 같은 숨김 항목 포함 여부
MAX_DEPTH = 5                # 재귀 탐색 깊이. None이면 제한 없음
TREE_MAX_ITEMS = 120         # 트리 미리보기 최대 항목 수
TABLE_MAX_ROWS = 500         # 상세 목록 출력 최대 행 수

TARGET_DIR

## 2. 스캔 함수

파일 크기, 수정 시간, 확장자, MIME 타입, 깊이를 수집합니다.

In [ ]:
def human_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.1f} {unit}" if unit != "B" else f"{int(size)} B"
        size /= 1024


def is_hidden(path: Path, root: Path) -> bool:
    try:
        parts = path.relative_to(root).parts
    except ValueError:
        parts = path.parts
    return any(part.startswith(".") for part in parts if part not in (".", ".."))


def depth_from_root(path: Path, root: Path) -> int:
    return len(path.relative_to(root).parts)


def scan_directory(root: Path, include_hidden: bool = True, max_depth: int | None = None) -> list[dict]:
    root = root.resolve()
    rows = []

    for current_root, dirnames, filenames in os.walk(root):
        current = Path(current_root)
        current_depth = depth_from_root(current, root) if current != root else 0

        if not include_hidden:
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            filenames = [f for f in filenames if not f.startswith(".")]

        if max_depth is not None and current_depth >= max_depth:
            dirnames[:] = []

        entries = [current / name for name in sorted(dirnames)] + [current / name for name in sorted(filenames)]
        for path in entries:
            if not include_hidden and is_hidden(path, root):
                continue
            try:
                stat = path.stat()
            except OSError as exc:
                rows.append({
                    "path": str(path.relative_to(root)),
                    "type": "error",
                    "error": str(exc),
                })
                continue

            is_file = path.is_file()
            mime, _ = mimetypes.guess_type(path.name)
            rows.append({
                "path": str(path.relative_to(root)),
                "name": path.name,
                "type": "file" if is_file else "dir" if path.is_dir() else "other",
                "extension": path.suffix.lower() if is_file else "",
                "size_bytes": stat.st_size if is_file else 0,
                "size": human_size(stat.st_size) if is_file else "",
                "modified": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
                "depth": depth_from_root(path, root),
                "mime": mime or "",
                "hidden": is_hidden(path, root),
            })

    return rows

## 3. 한 번에 요약 보기

아래 셀 하나로 전체 개요를 확인합니다.

In [ ]:
rows = scan_directory(TARGET_DIR, include_hidden=INCLUDE_HIDDEN, max_depth=MAX_DEPTH)

files = [row for row in rows if row.get("type") == "file"]
dirs = [row for row in rows if row.get("type") == "dir"]
total_size = sum(row["size_bytes"] for row in files)
extensions = Counter(row["extension"] or "[no extension]" for row in files)

print(f"대상 폴더: {TARGET_DIR}")
print(f"총 항목: {len(rows):,}개")
print(f"폴더: {len(dirs):,}개")
print(f"파일: {len(files):,}개")
print(f"파일 총 용량: {human_size(total_size)}")
print("\n확장자 TOP 10")
for ext, count in extensions.most_common(10):
    print(f"  {ext}: {count:,}개")

print("\n가장 큰 파일 TOP 10")
for row in sorted(files, key=lambda item: item["size_bytes"], reverse=True)[:10]:
    print(f"  {row['size']:>10}  {row['path']}")

print("\n최근 수정 파일 TOP 10")
for row in sorted(files, key=lambda item: item["modified"], reverse=True)[:10]:
    print(f"  {row['modified']}  {row['path']}")

## 4. 트리 미리보기

폴더 구조를 빠르게 훑어봅니다. 항목이 많으면 앞부분만 출력합니다.

In [ ]:
def print_tree(rows: list[dict], max_items: int = 120) -> None:
    shown = 0
    for row in sorted(rows, key=lambda item: item.get("path", "").lower()):
        if shown >= max_items:
            print(f"... {len(rows) - shown:,}개 항목 생략")
            break
        depth = row.get("depth", 1)
        icon = "[D]" if row.get("type") == "dir" else "[F]" if row.get("type") == "file" else "[?]"
        extra = f" ({row['size']})" if row.get("type") == "file" else ""
        indent = "  " * max(depth - 1, 0)
        print(f"{indent}{icon} {row.get('name', row.get('path'))}{extra}")
        shown += 1

print_tree(rows, TREE_MAX_ITEMS)

## 5. 상세 목록

Jupyter 환경에 `pandas`가 있으면 정렬 가능한 표로 보여주고, 없으면 기본 리스트로 출력합니다.

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(rows)
    display_columns = ["type", "path", "size", "modified", "extension", "mime", "hidden"]
    display(df[display_columns].head(TABLE_MAX_ROWS))
except ImportError:
    for row in rows[:TABLE_MAX_ROWS]:
        row_type = row.get("type", "")
        row_path = row.get("path", "")
        size = f" | {row.get('size')}" if row.get("size") else ""
        print(f"{row_type:>5} | {row_path}{size}")
    if len(rows) > TABLE_MAX_ROWS:
        print(f"... {len(rows) - TABLE_MAX_ROWS:,}개 항목 생략")

## 6. 필요할 때 검색하기

파일명 일부나 확장자로 빠르게 필터링할 수 있습니다.

In [ ]:
QUERY = ""       # 예: "readme", "license", ".py"
ONLY_FILES = False

query = QUERY.lower().strip()
matches = []
for row in rows:
    if ONLY_FILES and row.get("type") != "file":
        continue
    if not query or query in row.get("path", "").lower():
        matches.append(row)

print(f"검색어: {QUERY!r} / 결과: {len(matches):,}개")
for row in matches[:100]:
    size = f" ({row.get('size')})" if row.get("size") else ""
    print(f"{row.get('type'):>5}  {row.get('path')}{size}")
if len(matches) > 100:
    print(f"... {len(matches) - 100:,}개 결과 생략")